# 实验七：部署验收与常见问题

本实验用于最后验收 `04_YOLO_Edge_TBE_Ascend` 的完整链路：PyACL 加载 OM、CPU NMS baseline、Ascend C 自定义 NMS、ACLNN 调用验证、延迟对比与 MindStudio Profiling。

当前实验采用官方 Ascend C + ACLNN 调用路线，不要求提供 Python 模块 `yolo_nms_custom`。


## 1. 最终验收清单

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">项目</th>
      <th style="text-align: left;">通过标准</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">环境</td>
      <td style="text-align: left;"><code>npu-smi info</code> 可看到 Ascend310B4，CANN 使用 <code>/usr/local/Ascend/ascend-toolkit/8.0.RC1</code>。</td>
    </tr>
    <tr>
      <td style="text-align: left;">OM 推理</td>
      <td style="text-align: left;">PyACL 能加载 YOLO OM 并得到真实输出，不依赖 dry-run。</td>
    </tr>
    <tr>
      <td style="text-align: left;">CPU baseline</td>
      <td style="text-align: left;">CPU NMS 能输出参考检测结果和 latency。</td>
    </tr>
    <tr>
      <td style="text-align: left;">自定义算子</td>
      <td style="text-align: left;"><code>YoloNmsCustomProject</code> 能编译并安装 <code>custom_opp_ubuntu_aarch64.run</code>。</td>
    </tr>
    <tr>
      <td style="text-align: left;">ACLNN 接口</td>
      <td style="text-align: left;">能找到 <code>aclnn_yolo_nms_custom.h</code>，接口包含 <code>aclnnYoloNmsCustomGetWorkspaceSize</code> 和 <code>aclnnYoloNmsCustom</code>。</td>
    </tr>
    <tr>
      <td style="text-align: left;">NPU 验证</td>
      <td style="text-align: left;"><code>YoloNmsAclNNInvocation/bash run.sh</code> 输出 <code>PASSED</code>。</td>
    </tr>
    <tr>
      <td style="text-align: left;">Profiling</td>
      <td style="text-align: left;">MindStudio 或 <code>msprof</code> 能采集到自定义算子相关 task/kernel。</td>
    </tr>
  </tbody>
</table>


## 2. 一键检查关键文件

这一步不会修改任何文件，只检查当前开发板上的实验产物是否齐全。


In [ ]:
from pathlib import Path

ROOT = Path('/home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend')
paths = [
    ROOT / 'src/configs/yolo_edge.yaml',
    ROOT / 'models/yolov5s_310b4.om',
    ROOT / 'outputs/yolo_nms_inputs.npz',
    ROOT / 'src/operators/ascendc/YoloNmsCustomProject/build_out/custom_opp_ubuntu_aarch64.run',
    ROOT / 'src/operators/ascendc/YoloNmsAclNNInvocation/run.sh',
    ROOT / 'src/operators/ascendc/YoloNmsAclNNInvocation/src/main.cpp',
    ROOT / 'src/operators/ascendc/YoloNmsAclNNInvocation/src/op_runner.cpp',
    ROOT / 'src/operators/ascendc/YoloNmsAclNNInvocation/scripts/gen_data.py',
    ROOT / 'src/operators/ascendc/YoloNmsAclNNInvocation/scripts/verify_result.py',
]

for path in paths:
    print(('OK   ' if path.exists() else 'MISS '), path)


## 3. 检查 OPP 安装结果

如果这里找不到 `8.0.RC1/opp/vendors/customize` 下的文件，说明自定义算子没有安装到当前 CANN 环境，或者装到了旧的 7.0.0 / 7.0.RC1 目录。


In [ ]:
%%bash
source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1

echo "npu-smi:"
npu-smi info | head -20 || true

echo "CANN tools:"
which opc
which msopgen || true

echo "custom OPP files:"
find /usr/local/Ascend/ascend-toolkit/8.0.RC1/opp/vendors/customize   \( -name 'aclnn_yolo_nms_custom.h' -o -name '*YoloNmsCustom*' -o -name '*yolo_nms_custom*' \)   2>/dev/null | head -40


## 4. 检查 ACLNN 头文件签名

当前正确接口应该包含两个函数：`aclnnYoloNmsCustomGetWorkspaceSize` 和 `aclnnYoloNmsCustom`。


In [ ]:
%%bash
HEADER=/usr/local/Ascend/ascend-toolkit/8.0.RC1/opp/vendors/customize/op_api/include/aclnn_yolo_nms_custom.h
if [ ! -f "$HEADER" ]; then
  echo "missing: $HEADER"
  exit 0
fi

grep -n "aclnnYoloNmsCustomGetWorkspaceSize\|aclnnYoloNmsCustom" "$HEADER"


## 5. 重新跑一次 ACLNN 正确性验证

这是最终验收中最关键的一步。看到 `PASSED`，并且 `custom count`、`custom first 20` 与 `golden` 一致，就说明当前自定义算子链路可用。


In [ ]:
%%bash
set -e
cd /home/HwHiAiUser/samples/notebooks/YOLO_Edge_TBE_Ascend/src/operators/ascendc/YoloNmsAclNNInvocation
source /usr/local/Ascend/ascend-toolkit/8.0.RC1/aarch64-linux/script/set_env.sh
export ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1
bash run.sh


## 6. 常见问题

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">现象</th>
      <th style="text-align: left;">原因</th>
      <th style="text-align: left;">处理方式</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>No module named yolo_nms_custom</code></td>
      <td style="text-align: left;">没有做 Python 封装</td>
      <td style="text-align: left;">不影响当前实验。使用 <code>YoloNmsAclNNInvocation/bash run.sh</code> 验证。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>op_info</code> 或 tiling 报旧名字 <code>AddCustom</code></td>
      <td style="text-align: left;">工程从官方样例复制后没有改全，或 build 缓存没清理</td>
      <td style="text-align: left;">按官方“新建算子工程开发算子”重新生成工程，删除 <code>build_out/build</code> 后再编译。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>unexpected keyword argument mc2_ctx</code></td>
      <td style="text-align: left;">samples 分支和 CANN 版本不匹配</td>
      <td style="text-align: left;">CANN 8.0.RC1 对应使用 samples 8.0.RC1。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>Permission denied</code></td>
      <td style="text-align: left;">脚本没有执行权限，或从 Windows 复制后权限丢失</td>
      <td style="text-align: left;"><code>chmod +x run.sh scripts/*.sh</code>，必要时处理 CRLF。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>/bin/bash^M: bad interpreter</code></td>
      <td style="text-align: left;">文件是 Windows CRLF 换行</td>
      <td style="text-align: left;">`sed -i 's/</td>
    </tr>
    <tr>
      <td style="text-align: left;">$//' run.sh scripts/*.sh`。</td>
    </tr>
    <tr>
      <td style="text-align: left;">找不到输入 bin</td>
      <td style="text-align: left;">runner 的工作目录与脚本中的相对路径不一致</td>
      <td style="text-align: left;">当前修正路线使用 <code>../input</code>、<code>../output</code>，因为可执行程序在 <code>output/</code> 目录运行。</td>
    </tr>
    <tr>
      <td style="text-align: left;">boxes 数量不是 51</td>
      <td style="text-align: left;"><code>src/main.cpp</code> shape 固定为实验四验证样例</td>
      <td style="text-align: left;">使用同一份验证样例，或同步修改 <code>boxesShape/scoresShape/keepShape</code> 后重新编译。</td>
    </tr>
    <tr>
      <td style="text-align: left;">安装成功但找不到头文件</td>
      <td style="text-align: left;">装到了旧 CANN 目录</td>
      <td style="text-align: left;">确认 <code>ASCEND_CANN_PACKAGE_PATH=/usr/local/Ascend/ascend-toolkit/8.0.RC1</code>。</td>
    </tr>
  </tbody>
</table>


## 7. 实验报告建议结构

最终报告建议按这个顺序写：

1. 实验环境：Atlas 200I DK A2、Ascend310B4、CANN 8.0.RC1、npu-smi 23.0.rc3。
2. OM 推理：PyACL 加载 YOLO OM，得到 `pred_real`。
3. CPU baseline：说明 YOLO 后处理和 CPU NMS 的输入、输出、耗时。
4. 自定义算子：说明选择 Ascend C 路线，`YoloNmsCustom` 的输入输出、tiling、kernel、编译安装流程。
5. ACLNN 验证：贴出 `PASSED`、`custom count`、`golden count`、前 20 个索引。
6. 延迟对比：区分 CPU NMS 算法时间、ACLNN runner 端到端时间、MindStudio kernel/task 时间。
7. 结论：后处理 NMS 已从 CPU 路线迁移到 NPU 自定义算子路线，功能正确；性能分析以 Profiling 的算子级耗时为准。


## 8. 最终结论

如果实验五输出 `PASSED`，实验六采集到 Profiling，并且本 notebook 的检查项没有关键缺失，就可以认为 `04_YOLO_Edge_TBE_Ascend` 这一组实验完成。

当前项目不需要为了实验验收再补 Python `yolo_nms_custom` 封装。只有当后续课程要求“在 Python 中直接调用自定义 NMS 函数”时，才需要额外开发 pybind11/ctypes 封装层。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 最终验收时，哪一项最能说明自定义 NMS 功能正确？
   - A. custom keep/count 与 CPU baseline 一致并显示 PASSED
   - B. 只看到 CMake Warning
   - C. 只看到文件夹存在
   - D. 只执行了 git status

2. (单选题) 检查自定义 OPP 是否安装成功，通常会在哪个目录下查找？
   - A. /usr/local/Ascend/ascend-toolkit/.../opp/vendors/customize
   - B. /tmp/ipykernel
   - C. /home/user/.ssh
   - D. /mnt/c/Windows

3. (单选题) 如果报 `No such file or directory: YoloNmsAclNNInvocation`，最可能原因是？
   - A. 调用工程尚未创建或路径不一致
   - B. NPU 温度过高
   - C. CPU NMS 算法错误
   - D. bus.jpg 文件太小

4. (单选题) 如果 notebook 中出现 CMake Warning 但后续生成 .o/.json 和 run 包并安装 SUCCESS，应如何判断？
   - A. 优先看关键产物和 SUCCESS，warning 不一定是失败
   - B. 一定全部失败
   - C. 必须重装系统
   - D. 忽略所有输出

5. (多选题) 部署验收清单应包含哪些内容？
   - A. OM 模型可加载
   - B. CPU baseline 可生成
   - C. 自定义 OPP 已安装
   - D. ACLNN runner 输出 PASSED

6. (多选题) 常见问题中，哪些与路径或环境有关？
   - A. ASCEND_CANN_PACKAGE_PATH 指错
   - B. CANN set_env.sh 未 source
   - C. runner 工程目录不存在
   - D. input/output 文件路径不匹配

7. (多选题) 如果验证结果 mismatch，可以优先检查哪些项？
   - A. boxes/scores 是否同一份输入
   - B. 是否按 score 降序排序
   - C. main.cpp shape 是否与输入 N 一致
   - D. CPU 和 custom 的 iou_threshold/max_output 是否一致

8. (判断题) 最终验收只需要看到 `acl executable run success`，不需要比较输出。

9. (判断题) 实验零除外，后续各章节都可以通过课后练习回顾关键概念和排错方法。

10. (填空题) ACLNN 自定义算子头文件通常命名为 `____`。

11. (填空题) 最终正确性验证通过时，常见输出关键字是 `____`。

12. (简答题) 为什么部署验收要同时检查文件产物、安装位置和运行输出？

13. (简答题) 遇到 `ModuleNotFoundError: yolo_nms_custom` 是否一定说明实验失败？

14. (简答题) 如何把本实验结果写成严谨的最终结论？

15. (代码设计题) 写一组命令，快速检查自定义 OPP 安装和 ACLNN 头文件是否存在。

> 参考答案见 answer/04.08_deployment_validation_and_chapter_test_answer.ipynb。
